Creating a Data Cube from Combining All Dim and Fact Table

In [0]:
%sql
create or replace table gold_medical_data.data_cubes.medical_data_cube as 
with procedures_join as (
  select
    dp.encounter_id,
    dp.patient_id,
    dp.procedure_code,
    dp.procedure_code_description,
    fp.base_procedure_cost,
    fp.procedure_start,
    fp.procedure_stop
  from
    gold_medical_data.dim_tables.dim_procedures dp
      join gold_medical_data.fact_tables.fact_procedures fp
        on dp.encounter_id = fp.encounter_id
        and dp.patient_id = fp.patient_id
),
encounters_join as (
  select
    de.encounter_id,
    fe.patient_id,
    fe.payer_id,
    fe.organization_id,
    de.encounter_class,
    de.encounter_code,
    de.encounter_code_description,
    fe.base_encounter_cost,
    fe.total_claim_cost,
    fe.payer_coverage,
    fe.encounter_start,
    fe.encounter_stop
  from
    gold_medical_data.dim_tables.dim_encounters de
      join gold_medical_data.fact_tables.fact_encounters fe
        on de.encounter_id = fe.encounter_id
),
join_encounters_payers_organizations_patients as (
  select
    ej.encounter_id,
    ej.patient_id,
    ej.payer_id,
    ej.organization_id,
    ej.base_encounter_cost,
    ej.total_claim_cost,
    ej.payer_coverage,
    ej.encounter_start,
    ej.encounter_stop,
    ej.encounter_class,
    ej.encounter_code,
    ej.encounter_code_description,
    dp.payer_name,
    dp.payer_address,
    dp.payer_city,
    dp.payer_state,
    dp.payer_zip,
    do.organization_name,
    do.organization_address,
    do.organization_city,
    do.organization_state,
    do.organization_zip,
    do.organization_lat,
    do.organization_lon,
    dpa.birth_date,
    dpa.death_date,
    dpa.prefix,
    dpa.first_name,
    dpa.last_name,
    dpa.suffix,
    dpa.maiden,
    dpa.marital_status,
    dpa.race,
    dpa.ethnicity,
    dpa.gender,
    dpa.birth_place,
    dpa.address,
    dpa.city_name,
    dpa.state,
    dpa.county,
    dpa.zip_code,
    dpa.lat,
    dpa.lon
  from
    encounters_join ej
      join gold_medical_data.dim_tables.dim_payers dp
        on ej.payer_id = dp.payer_id
      join gold_medical_data.dim_tables.dim_organizations do
        on ej.organization_id = do.organization_id
      join gold_medical_data.dim_tables.dim_patients dpa
        on ej.patient_id = dpa.patient_id
)
select
  je.*,
  pj.procedure_code,
  pj.procedure_code_description,
  pj.procedure_start,
  pj.procedure_stop,
  pj.base_procedure_cost
from
  join_encounters_payers_organizations_patients je
    join procedures_join pj
      on je.encounter_id = pj.encounter_id
      and je.patient_id = pj.patient_id